# Feature Engineering

This notebook transforms the cleaned electricity consumption data into a
model-ready daily forecasting dataset.

Feature engineering is guided by findings from EDA and is designed to avoid
future-data leakage.

The forecasting target is daily electricity demand aggregated across the
observed households.

In [105]:
import duckdb
import pandas as pandas
con=duckdb.connect()

daily_features = con.execute(f"""
    SELECT
        CAST (DateTime AS DATE )AS date, 
        SUM(consumption_kwh) AS daily_kwh
    FROM read_parquet('../data/processed/lcl_cleaned.parquet')
    GROUP BY CAST(DateTime As DATE)
    ORDER BY date

""").df()

daily_features.head()


,date,daily_kwh
0,2011-12-06,2.947
1,2011-12-07,143.160
2,2011-12-08,248.374
3,2011-12-09,216.327
4,2011-12-10,246.168


In [106]:

daily_features.shape

(816, 2)

In [107]:

daily_features.dtypes

date         datetime64[us]
daily_kwh           float64
dtype: object

## Create the modeling window

Remember our EDA decision: the first and last days are incomplete boundary days.

So don't delete them from the processed Parquet, but exclude them from the modeling dataset.

In [108]:
model_df = daily_features.loc[
    (daily_features["date"] >= "2011-12-07") &
    (daily_features["date"] <= "2014-02-27")

].copy()

model_df.head()

,date,daily_kwh
1,2011-12-07,143.160
2,2011-12-08,248.374
3,2011-12-09,216.327
4,2011-12-10,246.168
5,2011-12-11,277.649


In [109]:
model_df.shape

(814, 2)

In [110]:
model_df.describe()

,date,daily_kwh
count,814,814.000000
mean,2013-01-16 12:00:00,294.026541
min,2011-12-07 00:00:00,142.655000
25%,2012-06-27 06:00:00,226.008000
50%,2013-01-16 12:00:00,270.280000
75%,2013-08-07 18:00:00,364.832750
max,2014-02-27 00:00:00,511.710000
std,NaN,88.215634


### Add calendar features

Now we'll create the first actual features from the patterns we established during EDA.


In [111]:
model_df["day_of_week"]=model_df["date"].dt.day_of_week
model_df["month"]=model_df["date"].dt.month

model_df.head()

,date,daily_kwh,day_of_week,month
1,2011-12-07,143.160,2,12
2,2011-12-08,248.374,3,12
3,2011-12-09,216.327,4,12
4,2011-12-10,246.168,5,12
5,2011-12-11,277.649,6,12


Why these two?

Because EDA gave us evidence for both:

Weekly pattern → day_of_week
Strong annual pattern → month

#### we'll handle the cyclical encoding properly before moving to the lag features.


## Cyclical Encoding:
Cyclical encoding transforms periodic time features (like hours, days of the week, or months) into continuous spatial coordinates using trigonometric functions—specifically sine ($\sin$) and cosine ($\cos$).

In [112]:
# Apply sine and cosine transformations
# df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
# df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)



import numpy as np
model_df["days_of_week_sin"]=np.sin(2 * np.pi *model_df['day_of_week'] / 7)
model_df["days_of_week_cos"]=np.cos(2 * np.pi *model_df['day_of_week'] / 7)

model_df["month_sin"]=np.sin(2 * np.pi *model_df['month'] / 12)
model_df["month_cos"]=np.cos(2 * np.pi *model_df['month'] / 12)


model_df.head()

,date,daily_kwh,day_of_week,month,days_of_week_sin,days_of_week_cos,month_sin,month_cos
1,2011-12-07,143.160,2,12,0.974928,-0.222521,-2.449294e-16,1.0
2,2011-12-08,248.374,3,12,0.433884,-0.900969,-2.449294e-16,1.0
3,2011-12-09,216.327,4,12,-0.433884,-0.900969,-2.449294e-16,1.0
4,2011-12-10,246.168,5,12,-0.974928,-0.222521,-2.449294e-16,1.0
5,2011-12-11,277.649,6,12,-0.781831,0.623490,-2.449294e-16,1.0


This is a good foundation because we're representing both weekly seasonality and annual seasonality without creating an artificial numerical distance between December → January or Sunday → Monday.

## Next task: Lag features

This is the most important feature-engineering step because our EDA showed very strong temporal persistence:

1-day lag: 0.9679

2-day lag: 0.9499  etc.

In [113]:
for lag in [1, 2, 3, 7]:
    model_df[f"lag_{lag}"]=model_df["daily_kwh"].shift(lag)
    
    print(f"Lag {lag:>2}")
model_df.head(10)

Lag  1
Lag  2
Lag  3
Lag  7


,date,daily_kwh,day_of_week,month,days_of_week_sin,days_of_week_cos,month_sin,month_cos,lag_1,lag_2,lag_3,lag_7
1,2011-12-07,143.160000,2,12,0.974928,-0.222521,-2.449294e-16,1.0,NaN,NaN,NaN,NaN
2,2011-12-08,248.374000,3,12,0.433884,-0.900969,-2.449294e-16,1.0,143.160000,NaN,NaN,NaN
3,2011-12-09,216.327000,4,12,-0.433884,-0.900969,-2.449294e-16,1.0,248.374000,143.160,NaN,NaN
4,2011-12-10,246.168000,5,12,-0.974928,-0.222521,-2.449294e-16,1.0,216.327000,248.374,143.160,NaN
5,2011-12-11,277.649000,6,12,-0.781831,0.623490,-2.449294e-16,1.0,246.168000,216.327,248.374,NaN
6,2011-12-12,266.042000,0,12,0.000000,1.000000,-2.449294e-16,1.0,277.649000,246.168,216.327,NaN
7,2011-12-13,230.629000,1,12,0.781831,0.623490,-2.449294e-16,1.0,266.042000,277.649,246.168,NaN
8,2011-12-14,268.984000,2,12,0.974928,-0.222521,-2.449294e-16,1.0,230.629000,266.042,277.649,143.160
9,2011-12-15,273.415001,3,12,0.433884,-0.900969,-2.449294e-16,1.0,268.984000,230.629,266.042,248.374
10,2011-12-16,262.446000,4,12,-0.433884,-0.900969,-2.449294e-16,1.0,273.415001,268.984,230.629,216.327


 So don't fill these NaNs with 0. Zero would incorrectly tell the model that demand was actually zero.

In [114]:
model_df[["lag_1","lag_2","lag_3","lag_7"]].isna().sum()

lag_1    1
lag_2    2
lag_3    3
lag_7    7
dtype: int64

In [115]:
model_df.shape

(814, 12)

Now make the modeling dataset

Because our largest lag is lag_7, only the first 7 observations lack the complete feature set.

We should not modify model_df yet. Instead, create a clean modeling dataframe

In [116]:
model_ready=model_df.dropna(
    subset=["lag_1","lag_2","lag_3","lag_7"]).copy()

model_ready.head()

,date,daily_kwh,day_of_week,month,days_of_week_sin,days_of_week_cos,month_sin,month_cos,lag_1,lag_2,lag_3,lag_7
8,2011-12-14,268.984000,2,12,0.974928,-0.222521,-2.449294e-16,1.0,230.629000,266.042000,277.649000,143.160
9,2011-12-15,273.415001,3,12,0.433884,-0.900969,-2.449294e-16,1.0,268.984000,230.629000,266.042000,248.374
10,2011-12-16,262.446000,4,12,-0.433884,-0.900969,-2.449294e-16,1.0,273.415001,268.984000,230.629000,216.327
11,2011-12-17,246.269000,5,12,-0.974928,-0.222521,-2.449294e-16,1.0,262.446000,273.415001,268.984000,246.168
12,2011-12-18,285.852000,6,12,-0.781831,0.623490,-2.449294e-16,1.0,246.269000,262.446000,273.415001,277.649


In [117]:
model_ready.shape

(807, 12)

In [118]:
model_ready.isna().sum()

date                0
daily_kwh           0
day_of_week         0
month               0
days_of_week_sin    0
days_of_week_cos    0
month_sin           0
month_cos           0
lag_1               0
lag_2               0
lag_3               0
lag_7               0
dtype: int64

zero missing values in these features.

Why we're doing this

Think of it like this:

814 daily observations
        ↓
Need previous 7 days
        ↓
First 7 cannot be used
        ↓
807 complete observations
        ↓
MODEL-READY DATA

We're deliberately removing only rows that cannot possibly have the required historical information, rather than inventing values

## rolling features
Create a simple rolling-demand feature using only past observations.

We'll start with a 7-day historical rolling mean, shifted by one day so today's target cannot influence today's feature

In [119]:
model_ready["rolling_mean_7"] = (model_ready["daily_kwh"]
                                .shift(1)
                                .rolling(7)
                                .mean())

In [120]:
model_ready[["date", "daily_kwh", "lag_1", "lag_7", "rolling_mean_7"]].head(10)

,date,daily_kwh,lag_1,lag_7,rolling_mean_7
8,2011-12-14,268.984000,230.629000,143.160000,NaN
9,2011-12-15,273.415001,268.984000,248.374000,NaN
10,2011-12-16,262.446000,273.415001,216.327000,NaN
11,2011-12-17,246.269000,262.446000,246.168000,NaN
12,2011-12-18,285.852000,246.269000,277.649000,NaN
13,2011-12-19,271.313000,285.852000,266.042000,NaN
14,2011-12-20,281.951999,271.313000,230.629000,NaN
15,2011-12-21,266.697000,281.951999,268.984000,270.033000
16,2011-12-22,220.936000,266.697000,273.415001,269.706286
17,2011-12-23,224.914000,220.936000,262.446000,262.209285


In [121]:
model_ready["rolling_mean_7"].isna().sum()

np.int64(7)

In [122]:
model_ready.shape

(807, 13)

In [123]:
model_ready.info()

<class 'pandas.DataFrame'>
RangeIndex: 807 entries, 8 to 814
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   date              807 non-null    datetime64[us]
 1   daily_kwh         807 non-null    float64       
 2   day_of_week       807 non-null    int32         
 3   month             807 non-null    int32         
 4   days_of_week_sin  807 non-null    float64       
 5   days_of_week_cos  807 non-null    float64       
 6   month_sin         807 non-null    float64       
 7   month_cos         807 non-null    float64       
 8   lag_1             807 non-null    float64       
 9   lag_2             807 non-null    float64       
 10  lag_3             807 non-null    float64       
 11  lag_7             807 non-null    float64       
 12  rolling_mean_7    800 non-null    float64       
dtypes: datetime64[us](1), float64(10), int32(2)
memory usage: 75.8 KB


In [124]:
model_ready.describe()

,date,daily_kwh,day_of_week,month,days_of_week_sin,days_of_week_cos,month_sin,month_cos,lag_1,lag_2,lag_3,lag_7,rolling_mean_7
count,807,807.000000,807.000000,807.000000,807.000000,807.000000,8.070000e+02,8.070000e+02,807.000000,807.000000,807.000000,807.000000,800.000000
mean,2013-01-20 00:00:00,294.559176,2.998761,6.278810,0.001746,-0.001392,4.438725e-02,7.077415e-02,294.423472,294.328846,294.267945,293.558069,294.406043
min,2011-12-14 00:00:00,142.655000,0.000000,1.000000,-0.974928,-0.900969,-1.000000e+00,-1.000000e+00,142.655000,142.655000,142.655000,142.655000,159.905286
25%,2012-07-02 12:00:00,226.020000,1.000000,3.000000,-0.781831,-0.900969,-5.000000e-01,-5.000000e-01,226.020000,226.020000,226.020000,225.797500,227.798500
50%,2013-01-20 00:00:00,270.449000,3.000000,6.000000,0.000000,-0.222521,1.224647e-16,6.123234e-17,270.401000,270.302000,270.302000,269.648000,269.687286
75%,2013-08-09 12:00:00,365.487999,5.000000,9.500000,0.781831,0.623490,8.660254e-01,8.660254e-01,365.487999,365.487999,365.487999,365.001000,366.923286
max,2014-02-27 00:00:00,511.710000,6.000000,12.000000,0.974928,1.000000,1.000000e+00,1.000000e+00,511.710000,511.710000,511.710000,511.710000,488.802000
std,NaN,88.328090,1.999069,3.641549,0.707665,0.707422,6.961967e-01,7.138368e-01,88.342096,88.331535,88.326065,88.436929,86.586732


In [125]:
model_ready.head()

,date,daily_kwh,day_of_week,month,days_of_week_sin,days_of_week_cos,month_sin,month_cos,lag_1,lag_2,lag_3,lag_7,rolling_mean_7
8,2011-12-14,268.984000,2,12,0.974928,-0.222521,-2.449294e-16,1.0,230.629000,266.042000,277.649000,143.160,NaN
9,2011-12-15,273.415001,3,12,0.433884,-0.900969,-2.449294e-16,1.0,268.984000,230.629000,266.042000,248.374,NaN
10,2011-12-16,262.446000,4,12,-0.433884,-0.900969,-2.449294e-16,1.0,273.415001,268.984000,230.629000,216.327,NaN
11,2011-12-17,246.269000,5,12,-0.974928,-0.222521,-2.449294e-16,1.0,262.446000,273.415001,268.984000,246.168,NaN
12,2011-12-18,285.852000,6,12,-0.781831,0.623490,-2.449294e-16,1.0,246.269000,262.446000,273.415001,277.649,NaN


In [126]:
model_ready = model_ready.dropna(
    subset=["rolling_mean_7"]
).copy()



In [127]:
model_ready.head()

,date,daily_kwh,day_of_week,month,days_of_week_sin,days_of_week_cos,month_sin,month_cos,lag_1,lag_2,lag_3,lag_7,rolling_mean_7
15,2011-12-21,266.697,2,12,0.974928,-0.222521,-2.449294e-16,1.0,281.951999,271.313000,285.852000,268.984000,270.033000
16,2011-12-22,220.936,3,12,0.433884,-0.900969,-2.449294e-16,1.0,266.697000,281.951999,271.313000,273.415001,269.706286
17,2011-12-23,224.914,4,12,-0.433884,-0.900969,-2.449294e-16,1.0,220.936000,266.697000,281.951999,262.446000,262.209285
18,2011-12-24,225.996,5,12,-0.974928,-0.222521,-2.449294e-16,1.0,224.914000,220.936000,266.697000,246.269000,256.847571
19,2011-12-25,225.718,6,12,-0.781831,0.623490,-2.449294e-16,1.0,225.996000,224.914000,220.936000,285.852000,253.951428


In [128]:
model_ready.info()

<class 'pandas.DataFrame'>
RangeIndex: 800 entries, 15 to 814
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   date              800 non-null    datetime64[us]
 1   daily_kwh         800 non-null    float64       
 2   day_of_week       800 non-null    int32         
 3   month             800 non-null    int32         
 4   days_of_week_sin  800 non-null    float64       
 5   days_of_week_cos  800 non-null    float64       
 6   month_sin         800 non-null    float64       
 7   month_cos         800 non-null    float64       
 8   lag_1             800 non-null    float64       
 9   lag_2             800 non-null    float64       
 10  lag_3             800 non-null    float64       
 11  lag_7             800 non-null    float64       
 12  rolling_mean_7    800 non-null    float64       
dtypes: datetime64[us](1), float64(10), int32(2)
memory usage: 75.1 KB


So Feature Engineering is ready to be finalized.

Step 5 — Save the engineered dataset

Before moving to modeling, save this as a reproducible artifact. into .parquetfile

In [130]:
model_ready.to_parquet(
    "../data/features/daily_model_features.parquet",
    index=False
)

In [ ]:
model_ready.shape

## Why save this?

We now have a clean separation:

RAW DATA
   ↓
ETL / CLEANING
   ↓
lcl_cleaned.parquet
   ↓
EDA
   ↓
Feature Engineering
   ↓
daily_model_features.parquet
   ↓
05_modeling.ipynb


### So this is a real milestone:

Raw data → validated → cleaned → explored → transformed into model-ready features.